# 03 — SQL Analysis — Complete

## Purpose
This notebook contains the project's **21 SQL business queries** from the original SQL script, adapted for the workflow.

### Analytical workflow
For each query:

**Business question → SQL technique → result → interpretation**

The separate `.sql` file is kept in the project's `sql/` folder. This notebook provides executable SQL analysis and interpretation.

> The dataset is synthetic and represents a simulated Zimbabwean trucking operation.


In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

PROJECT = Path.cwd().parent
DATA_DIR = PROJECT / "data" / "processed"
DB_PATH = PROJECT / "data" / "fleet_analysis.db"

conn = sqlite3.connect(DB_PATH)

print("Project:", PROJECT)
print("Database:", DB_PATH)


In [ ]:
# Load the five cleaned datasets into SQLite tables.

tables = {
    "trips": "trips_clean.csv",
    "trucks": "trucks_clean.csv",
    "drivers": "drivers_clean.csv",
    "maintenance": "maintenance_clean.csv",
    "fuel_purchases": "fuel_purchases_clean.csv",
}

for table_name, file_name in tables.items():
    df = pd.read_csv(DATA_DIR / file_name)
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"{table_name}: {len(df):,} rows loaded")


## Q1. Total Trucks

**Section:** SECTION 1: FLEET OVERVIEW

**Business question:** What does this query tell us about the trucking operation?

**SQL technique:** COUNT / SUM / ROUND.


In [ ]:
query = '''
SELECT COUNT(*) as total_trucks FROM trucks;
'''

result = pd.read_sql_query(query, conn)
result


**Interpretation:** Establishes the size of the simulated fleet.


## Q2. Total Trips

**Section:** SECTION 1: FLEET OVERVIEW

**Business question:** What does this query tell us about the trucking operation?

**SQL technique:** COUNT / SUM / ROUND.


In [ ]:
query = '''
SELECT COUNT(*) as total_trips FROM trips;
'''

result = pd.read_sql_query(query, conn)
result


**Interpretation:** Counts the operational trips available for analysis.


## Q3. Total Distance

**Section:** SECTION 1: FLEET OVERVIEW

**Business question:** What does this query tell us about the trucking operation?

**SQL technique:** COUNT / SUM / ROUND.


In [ ]:
query = '''
SELECT ROUND(SUM(distance_km), 0) as total_distance_km FROM trips;
'''

result = pd.read_sql_query(query, conn)
result


**Interpretation:** Shows the total distance covered by the simulated fleet.


## Q4. Total Revenue

**Section:** SECTION 1: FLEET OVERVIEW

**Business question:** What does this query tell us about the trucking operation?

**SQL technique:** COUNT / SUM / ROUND.


In [ ]:
query = '''
SELECT ROUND(SUM(revenue), 2) as total_revenue FROM trips;
'''

result = pd.read_sql_query(query, conn)
result


**Interpretation:** Measures the total revenue generated by all simulated trips.


## Q5. Total Profit

**Section:** SECTION 1: FLEET OVERVIEW

**Business question:** What does this query tell us about the trucking operation?

**SQL technique:** COUNT / SUM / ROUND.


In [ ]:
query = '''
SELECT ROUND(SUM(profit), 2) as total_profit FROM trips;
'''

result = pd.read_sql_query(query, conn)
result


**Interpretation:** Measures total profit after the operating costs recorded in the trip data.


## Q6. Top 5 Trucks by Revenue

**Section:** SECTION 2: TRUCK PERFORMANCE

**Business question:** What does this query tell us about the trucking operation?

**SQL technique:** JOIN + GROUP BY + ORDER BY.


In [ ]:
query = '''
SELECT t.truck_id, t.model, ROUND(SUM(trips.revenue), 2) as total_revenue
FROM trucks t LEFT JOIN trips ON t.truck_id = trips.truck_id
GROUP BY t.truck_id, t.model ORDER BY total_revenue DESC LIMIT 5;
'''

result = pd.read_sql_query(query, conn)
result


**Interpretation:** Identifies trucks contributing the most revenue.


## Q7. Top 5 Trucks by Trip Count

**Section:** SECTION 2: TRUCK PERFORMANCE

**Business question:** What does this query tell us about the trucking operation?

**SQL technique:** JOIN + GROUP BY + ORDER BY.


In [ ]:
query = '''
SELECT t.truck_id, t.model, COUNT(trips.trip_id) as trip_count
FROM trucks t LEFT JOIN trips ON t.truck_id = trips.truck_id
GROUP BY t.truck_id, t.model ORDER BY trip_count DESC LIMIT 5;
'''

result = pd.read_sql_query(query, conn)
result


**Interpretation:** Shows which trucks are used most frequently.


## Q8. Top 5 Trucks by Fuel Efficiency

**Section:** SECTION 2: TRUCK PERFORMANCE

**Business question:** What does this query tell us about the trucking operation?

**SQL technique:** JOIN + GROUP BY + ORDER BY.


In [ ]:
query = '''
SELECT t.truck_id, t.model,
       ROUND(SUM(trips.distance_km) / NULLIF(SUM(trips.fuel_consumed_litres), 0), 2) AS fuel_efficiency
FROM trucks t
LEFT JOIN trips ON t.truck_id = trips.truck_id
GROUP BY t.truck_id, t.model
HAVING SUM(trips.fuel_consumed_litres) > 0
ORDER BY fuel_efficiency DESC LIMIT 5;
'''

result = pd.read_sql_query(query, conn)
result


**Interpretation:** Compares descriptive fuel efficiency across trucks. This is suitable for reporting, but fuel efficiency should not be used as an input when predicting fuel consumed because it is derived from distance and fuel.


## Q9. Top 5 Trucks by Maintenance Cost

**Section:** SECTION 2: TRUCK PERFORMANCE

**Business question:** What does this query tell us about the trucking operation?

**SQL technique:** JOIN + GROUP BY + ORDER BY.


In [ ]:
query = '''
SELECT t.truck_id, t.model, ROUND(SUM(m.cost), 2) as total_maintenance_cost
FROM trucks t LEFT JOIN maintenance m ON t.truck_id = m.truck_id
GROUP BY t.truck_id, t.model ORDER BY total_maintenance_cost DESC LIMIT 5;
'''

result = pd.read_sql_query(query, conn)
result


**Interpretation:** Identifies trucks with the highest accumulated maintenance spending.


## Q10. Top 5 Trucks by Profit

**Section:** SECTION 2: TRUCK PERFORMANCE

**Business question:** What does this query tell us about the trucking operation?

**SQL technique:** JOIN + GROUP BY + ORDER BY.


In [ ]:
query = '''
SELECT t.truck_id, t.model, ROUND(SUM(trips.profit), 2) as total_profit
FROM trucks t LEFT JOIN trips ON t.truck_id = trips.truck_id
GROUP BY t.truck_id, t.model ORDER BY total_profit DESC LIMIT 5;
'''

result = pd.read_sql_query(query, conn)
result


**Interpretation:** Identifies trucks contributing the most total profit.


## Q11. Top 5 Drivers by Deliveries

**Section:** SECTION 3: DRIVER PERFORMANCE

**Business question:** What does this query tell us about the trucking operation?

**SQL technique:** JOIN + GROUP BY + ORDER BY.


In [ ]:
query = '''
SELECT d.driver_id, d.full_name, COUNT(trips.trip_id) as delivery_count
FROM drivers d LEFT JOIN trips ON d.driver_id = trips.driver_id
GROUP BY d.driver_id, d.full_name ORDER BY delivery_count DESC LIMIT 5;
'''

result = pd.read_sql_query(query, conn)
result


**Interpretation:** Shows drivers with the highest number of recorded deliveries.


## Q12. Top 5 Drivers by Revenue

**Section:** SECTION 3: DRIVER PERFORMANCE

**Business question:** What does this query tell us about the trucking operation?

**SQL technique:** JOIN + GROUP BY + ORDER BY.


In [ ]:
query = '''
SELECT d.driver_id, d.full_name, ROUND(SUM(trips.revenue), 2) as total_revenue
FROM drivers d LEFT JOIN trips ON d.driver_id = trips.driver_id
GROUP BY d.driver_id, d.full_name ORDER BY total_revenue DESC LIMIT 5;
'''

result = pd.read_sql_query(query, conn)
result


**Interpretation:** Compares total revenue associated with each driver.


## Q13. Top 5 Drivers by Fuel Efficiency

**Section:** SECTION 3: DRIVER PERFORMANCE

**Business question:** What does this query tell us about the trucking operation?

**SQL technique:** JOIN + GROUP BY + ORDER BY.


In [ ]:
query = '''
SELECT d.driver_id, d.full_name,
       ROUND(SUM(trips.distance_km) / NULLIF(SUM(trips.fuel_consumed_litres), 0), 2) AS fuel_efficiency
FROM drivers d
LEFT JOIN trips ON d.driver_id = trips.driver_id
GROUP BY d.driver_id, d.full_name
HAVING SUM(trips.fuel_consumed_litres) > 0
ORDER BY fuel_efficiency DESC LIMIT 5;
'''

result = pd.read_sql_query(query, conn)
result


**Interpretation:** Compares descriptive average fuel efficiency across drivers.


## Q14. Top 5 Drivers by Fewest Delays

**Section:** SECTION 3: DRIVER PERFORMANCE

**Business question:** What does this query tell us about the trucking operation?

**SQL technique:** JOIN + GROUP BY + ORDER BY.


In [ ]:
query = '''
SELECT d.driver_id, d.full_name, 
       COUNT(CASE WHEN trips.has_delay = 1 THEN 1 END) as total_delays,
       ROUND(CAST(COUNT(CASE WHEN trips.has_delay = 1 THEN 1 END) AS FLOAT) / 
             NULLIF(COUNT(trips.trip_id), 0) * 100, 2) as delay_rate_pct
FROM drivers d LEFT JOIN trips ON d.driver_id = trips.driver_id
GROUP BY d.driver_id, d.full_name HAVING COUNT(trips.trip_id) > 0
ORDER BY delay_rate_pct ASC LIMIT 5;
'''

result = pd.read_sql_query(query, conn)
result


**Interpretation:** Identifies drivers with the lowest observed delay rates among drivers with recorded trips.


## Q15. Top 5 Drivers by Profit

**Section:** SECTION 3: DRIVER PERFORMANCE

**Business question:** What does this query tell us about the trucking operation?

**SQL technique:** JOIN + GROUP BY + ORDER BY.


In [ ]:
query = '''
SELECT d.driver_id, d.full_name, ROUND(SUM(trips.profit), 2) as total_profit
FROM drivers d LEFT JOIN trips ON d.driver_id = trips.driver_id
GROUP BY d.driver_id, d.full_name ORDER BY total_profit DESC LIMIT 5;
'''

result = pd.read_sql_query(query, conn)
result


**Interpretation:** Compares total profit associated with each driver.


## Q16. Top 5 Routes by Revenue

**Section:** SECTION 4: ROUTE ANALYSIS (CTEs)

**Business question:** What does this query tell us about the trucking operation?

**SQL technique:** GROUP BY / CTE / aggregation.


In [ ]:
query = '''
SELECT route, ROUND(SUM(revenue), 2) as total_revenue
FROM trips GROUP BY route ORDER BY total_revenue DESC LIMIT 5;
'''

result = pd.read_sql_query(query, conn)
result


**Interpretation:** Identifies routes generating the highest revenue.


## Q17. Top 5 Routes by Profit (CTE)

**Section:** SECTION 4: ROUTE ANALYSIS (CTEs)

**Business question:** What does this query tell us about the trucking operation?

**SQL technique:** GROUP BY / CTE / aggregation.


In [ ]:
query = '''
WITH route_metrics AS (
    SELECT route, ROUND(SUM(revenue), 2) as total_revenue, ROUND(SUM(profit), 2) as total_profit
    FROM trips GROUP BY route
)
SELECT route, total_profit, ROUND((total_profit/total_revenue)*100, 2) as profit_margin_pct
FROM route_metrics WHERE total_revenue > 0 ORDER BY total_profit DESC LIMIT 5;
'''

result = pd.read_sql_query(query, conn)
result


**Interpretation:** Uses a CTE to calculate route-level revenue, profit and profit margin.


## Q18. Loss-Making Routes (CTE)

**Section:** SECTION 4: ROUTE ANALYSIS (CTEs)

**Business question:** What does this query tell us about the trucking operation?

**SQL technique:** GROUP BY / CTE / aggregation.


In [ ]:
query = '''
WITH route_metrics AS (
    SELECT route, ROUND(SUM(revenue), 2) as total_revenue, ROUND(SUM(profit), 2) as total_profit
    FROM trips GROUP BY route
)
SELECT route, total_revenue, total_profit
FROM route_metrics WHERE total_profit < 0 ORDER BY total_profit ASC;
'''

result = pd.read_sql_query(query, conn)
result


**Interpretation:** Uses a CTE to identify routes whose aggregated profit is negative.


## Q19. Monthly Revenue Trend (LAG)

**Section:** SECTION 5: TIME-BASED ANALYSIS (Window Functions)

**Business question:** What does this query tell us about the trucking operation?

**SQL technique:** Date aggregation + window functions.


In [ ]:
query = '''
WITH monthly_revenue AS (
    SELECT strftime('%Y-%m', trip_date) as month, ROUND(SUM(revenue), 2) as monthly_revenue
    FROM trips GROUP BY strftime('%Y-%m', trip_date) ORDER BY month
)
SELECT month, monthly_revenue,
       LAG(monthly_revenue, 1) OVER (ORDER BY month) as previous_month,
       ROUND(((monthly_revenue - LAG(monthly_revenue, 1) OVER (ORDER BY month)) / 
              NULLIF(LAG(monthly_revenue, 1) OVER (ORDER BY month), 0)) * 100, 2) as growth_pct
FROM monthly_revenue;
'''

result = pd.read_sql_query(query, conn)
result


**Interpretation:** Uses the LAG window function to compare each month's revenue with the previous month.


## Q20. Top Months by Revenue

**Section:** SECTION 5: TIME-BASED ANALYSIS (Window Functions)

**Business question:** What does this query tell us about the trucking operation?

**SQL technique:** Date aggregation + window functions.


In [ ]:
query = '''
SELECT strftime('%Y-%m', trip_date) as month, ROUND(SUM(revenue), 2) as total_revenue
FROM trips GROUP BY strftime('%Y-%m', trip_date) ORDER BY total_revenue DESC LIMIT 6;
'''

result = pd.read_sql_query(query, conn)
result


**Interpretation:** Identifies the highest-revenue months.


## Q21. Monthly Profit with Rolling Average

**Section:** SECTION 5: TIME-BASED ANALYSIS (Window Functions)

**Business question:** What does this query tell us about the trucking operation?

**SQL technique:** Date aggregation + window functions.


In [ ]:
query = '''
WITH monthly_profit AS (
    SELECT strftime('%Y-%m', trip_date) as month, ROUND(SUM(profit), 2) as monthly_profit
    FROM trips GROUP BY strftime('%Y-%m', trip_date) ORDER BY month
)
SELECT month, monthly_profit,
       ROUND(AVG(monthly_profit) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) as rolling_3month_avg
FROM monthly_profit;
'''

result = pd.read_sql_query(query, conn)
result


**Interpretation:** Uses a three-month rolling average to smooth monthly profit and reveal the underlying trend.


## Final reconciliation

These checks make sure the SQL database agrees with the cleaned trip data.


In [ ]:
checks = {
    "Trips": "SELECT COUNT(*) AS value FROM trips",
    "Revenue": "SELECT ROUND(SUM(revenue), 2) AS value FROM trips",
    "Operating cost": "SELECT ROUND(SUM(operating_cost), 2) AS value FROM trips",
    "Profit": "SELECT ROUND(SUM(profit), 2) AS value FROM trips",
    "Maintenance cost": "SELECT ROUND(SUM(cost), 2) AS value FROM maintenance",
}

for name, query in checks.items():
    value = pd.read_sql_query(query, conn).iloc[0, 0]
    print(f"{name}: {value}")

conn.close()


# Key learning points

By completing this notebook, you should be able to explain:

1. How CSV data is loaded into relational tables.
2. How `SELECT`, `WHERE`, `GROUP BY`, `HAVING` and `ORDER BY` answer business questions.
3. How `JOIN` combines fleet, driver, trip and maintenance information.
4. How CTEs make multi-step SQL analysis easier to read.
5. How `LAG()` and rolling window functions support time-series analysis.
6. Why SQL results need business interpretation.
7. Why descriptive SQL analysis is followed by statistical analysis and machine learning.

### Important distinction

The `.sql` file is the **SQL source/reference**.

This notebook is the **SQL execution and interpretation layer**.

Both should remain in the final project.
